<a href="https://colab.research.google.com/github/Anikrai11/My_All_Project/blob/main/Copy_of_Rag_analysis_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf  sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.7 MB/s eta 0:00:00


In [2]:
import os
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader
import numpy as np
from openai import OpenAI

In [3]:

embed_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
from transformers import pipeline
generator=pipeline("text-generation",
                   model="google/flan-t5-large",
                   device=0)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

# step_1

In [5]:
def load_pdf(pdf_path):
  reader=PdfReader(pdf_path)
  text=""
  for i,page in enumerate(reader.pages):
    text+=f"\n[page{i+1}]\n"+page.extract_text()
  return text

# step_2

In [6]:
def split_text(text,chunk_size=500,overloop=50):
  words=text.split()
  chunks=[]
  for i in range(0,len(words),chunk_size-overloop):

   chunk=" ".join(words[i:i+chunk_size])
   chunks.append(chunk)
  return chunks

# step_3

In [7]:
def create_index(chunks):
  embeddings=embed_model.encode(chunks,show_progress_bar=True)
  dimension=embeddings.shape[1]
  index=faiss.IndexFlatL2(dimension)
  index.add(np.array(embeddings))
  return index,embeddings


# retriver for question

#step_4

In [8]:
def retrieve(query,chunks,index,k=3):
  query_vector=embed_model.encode([query])
  D,I=index.search(np.array(query_vector),k)
  result=[]
  for i in I[0]:
    result.append(chunks[i])
  return result


# step_5

In [9]:
def generate_answer(query, context):
    prompt = f"""নিচের Context ব্যবহার করে প্রশ্নের উত্তর দাও।
যদি Context এ উত্তর না থাকে তাহলে বলো "আমি জানি না"।

Context:
{context}

প্রশ্ন: {query}
উত্তর:"""

    result = generator(prompt, max_length=300)
    return result[0]['generated_text']

# step_6

In [ ]:
if __name__== "__main__":
  pdf_path="/content/7 habits.pdf"
  print("1.pdf_loading")
  text=load_pdf(pdf_path)

  print("2.spliting")
  chunks=split_text(text)
  print(f"Total_chunks:{len(chunks)}")
  print("3.creating vactor index")
  index,embeddings=create_index(chunks)
  while True:
    query=input("\nyour question")
    if query=="exit":break
    print("searching")
    result=retrieve(query,chunks,index)
    context="\n---\n".join(result)
    print("genarating")
    answer=generate_answer(query,context)
    print(answer)
    print("\ndd",result[0][:300],"---")



1.pdf_loading
2.spliting
Total_chunks:276
3.creating vactor index


Batches:   0%|          | 0/9 [00:00<?, ?it/s]


your questionThe Quadrant II Approach


[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


searching
genarating


[transformers] Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


নিচের Context ব্যবহার করে প্রশ্নের উত্তর দাও।
যদি Context এ উত্তর না থাকে তাহলে বলো "আমি জানি না"।

Context:
Quadrant II life-style. The first-generation notepads and “to do” lists give us no more than a place to capture those things that penetrate our awareness so we won’t forget them. The second-generation appointment books and calendars merely provide a place to record our future commitments so that we can be where we have agreed to be at the appropriate time. Even the third generation, with its vast array of planners and materials, focuses primarily on helping people prioritize and plan their Quadrants I and III activities. Though many trainers and consultants recognize the value of Quadrant II activities, the actual planning tools of the third generation do not facilitate organizing and executing around them. As each generation builds on those that have preceded it, the strengths and some of the tools of each of the first three generations provide elemental material for the fourth